# 01 - Data loading and exploratory data analysis

Load raw backlink pricing data and explore distributions of price, quality metrics (DR, CF, TF), traffic, country, and TLD.

In [ ]:
from backlink_pricing_model.core.notebook import init_notebook
from backlink_pricing_model.preprocessing.data_loading import (
    load_raw_parquet,
)
from backlink_pricing_model.preprocessing.data_quality import (
    filter_valid_prices,
    missing_value_report,
    validate_metric_ranges,
)
from backlink_pricing_model.preprocessing.feature_engineering import (
    add_log_price,
    add_log_traffic,
    add_tld_feature,
    normalize_country,
    normalize_link_source_type,
    normalize_link_type,
)
from backlink_pricing_model.visualization.distributions_plots import (
    plot_country_distribution,
    plot_metric_distributions,
    plot_missing_values,
    plot_price_by_quality_tier,
    plot_price_by_tld,
    plot_price_distribution,
    plot_tld_distribution,
)

init_notebook()

## 1. Load raw data

Load the extracted backlink dataset from `data/raw/`. If you haven't run the extraction pipeline yet:

```bash
python -m scripts.data_pipeline.main
```

In [ ]:
df_raw = load_raw_parquet()
print(f"Loaded {len(df_raw):,} rows, {len(df_raw.columns)} columns")
df_raw.head()

In [ ]:
df_raw.info()

## 2. Missing value analysis

In [ ]:
report = missing_value_report(df_raw)
report

In [ ]:
plot_missing_values(df_raw).show()

## 3. Data cleaning

Filter valid prices, validate metric ranges, and normalize categorical fields.

In [ ]:
# Filter to valid price range and validate metrics.
df = filter_valid_prices(df_raw, min_price=0.0, max_price=50_000.0)
df = validate_metric_ranges(df)

# Normalize categorical columns.
df = normalize_country(df)
df = normalize_link_type(df)
df = normalize_link_source_type(df)

# Extract TLD from domain.
df = add_tld_feature(df)

# Add derived features.
df = add_log_price(df)
df = add_log_traffic(df)

print(f"Clean dataset: {len(df):,} rows")
df.describe()

## 4. Price distribution

In [ ]:
plot_price_distribution(df, log_scale=True).show()

In [ ]:
plot_price_distribution(df, log_scale=False).show()

## 5. Quality metric distributions (DR, CF, TF)

In [ ]:
plot_metric_distributions(df).show()

## 6. Price by quality tier

In [ ]:
for metric in ["dr", "cf", "tf"]:
    plot_price_by_quality_tier(df, metric=metric).show()

## 7. TLD analysis

In [ ]:
plot_tld_distribution(df, top_n=15).show()

In [ ]:
plot_price_by_tld(df, top_n=10).show()

## 8. Country distribution

In [ ]:
plot_country_distribution(df, top_n=15).show()

## 9. Summary statistics by category

In [ ]:
# Price stats by link source type.
source_stats = (
    df.groupby("link_source_type")["final_price"]
    .agg(["count", "mean", "median"])
    .round(2)
    .sort_values("count", ascending=False)
)
source_stats.head(10)

## 10. Save cleaned data for next notebook

In [ ]:
from backlink_pricing_model.preprocessing.data_loading import save_processed

path = save_processed(df, "backlinks_cleaned", subdir="processed")
print(f"Saved cleaned dataset to {path}")